# 00 · Rebuild the uploaded data files
`Genes.csv` (70 MB) and `Genes(479).csv` (42 MB) were uploaded to `data/raw/` as compressed pieces (`*.gz.part00`, `part01`, …) because the upload tool takes at most 10 MB at a time.
This notebook joins the pieces, unzips them, and checks each result against the original file's MD5 checksum.
**Run once**, before `01_load_and_qc.ipynb`. If you upload the full CSVs yourself, you don't need it.

In [ ]:
# ---- SETUP CELL ----
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/BT3041_Sepsis"
except ImportError:
    BASE = os.environ.get("BT3041_BASE", os.path.abspath(".."))
RAW = os.path.join(BASE, "data", "raw")
os.makedirs(RAW, exist_ok=True)

# move uploaded data files from the project folder into data/raw/
import shutil
for f in os.listdir(BASE):
    if ".gz.part" in f or f == "Patients.csv":
        shutil.move(os.path.join(BASE, f), os.path.join(RAW, f))
print("raw files:", sorted(os.listdir(RAW)))

In [ ]:
import glob, gzip, hashlib, io

EXPECTED_MD5 = {"Genes.csv": "f9e08cc188ac7b7357c90a46a44a64cc",
                "Genes(479).csv": "a895675cb340943daa2e48379af124ff",
                "Patients.csv": "9a7bcca2230a7d89083ebb91e8de2620"}

for name in ["Genes.csv", "Genes(479).csv"]:
    parts = sorted(glob.glob(os.path.join(RAW, glob.escape(name) + ".gz.part*")))
    print(name, "<-", [os.path.basename(p) for p in parts])
    joined = b"".join(open(p, "rb").read() for p in parts)
    with open(os.path.join(RAW, name), "wb") as out:
        out.write(gzip.decompress(joined))

for name, md5 in EXPECTED_MD5.items():
    got = hashlib.md5(open(os.path.join(RAW, name), "rb").read()).hexdigest()
    print(f"{name:16s} {os.path.getsize(os.path.join(RAW, name)):>11,d} bytes  checksum {'OK' if got == md5 else 'MISMATCH ' + got}")

Expected: three lines ending in `checksum OK` (Genes.csv 73,073,117 bytes; Genes(479).csv 43,676,104; Patients.csv 102,564).
After this you can delete the `*.gz.part*` files from `data/raw/`.